# Link to the dataset

https://data.tii.ie/Datasets/TrafficCountData/2013/06/01/per-vehicle-records-2013-06-01.csv


# Importing Requried Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

We started by importing all the necessary libraries for data handling and visualization.

# Loading the dataset

In [ ]:
taa = pd.read_csv('per-vehicle-records-2013-06-01(1).csv', engine = 'python', on_bad_lines='skip')

# Checking Dataset Structure

In [ ]:
taa.shape

(4769072, 26)

In [ ]:
taa.columns

Index(['cosit', 'year', 'month', 'day', 'hour', 'minute', 'second',
       'millisecond', 'minuteofday', 'lane', 'lanename', 'straddlelane',
       'straddlelanename', 'class', 'classname', 'length', 'headway', 'gap',
       'speed', 'weight', 'temperature', 'duration', 'validitycode',
       'numberofaxles', 'axleweights', 'axlespacings'],
      dtype='object')

# Checking Dataset Information

In [ ]:
taa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4769072 entries, 0 to 4769071
Data columns (total 26 columns):
 #   Column            Dtype  
---  ------            -----  
 0   cosit             int64  
 1   year              int64  
 2   month             int64  
 3   day               int64  
 4   hour              int64  
 5   minute            int64  
 6   second            int64  
 7   millisecond       int64  
 8   minuteofday       int64  
 9   lane              int64  
 10  lanename          object 
 11  straddlelane      int64  
 12  straddlelanename  float64
 13  class             int64  
 14  classname         object 
 15  length            float64
 16  headway           float64
 17  gap               float64
 18  speed             float64
 19  weight            float64
 20  temperature       float64
 21  duration          int64  
 22  validitycode      int64  
 23  numberofaxles     int64  
 24  axleweights       object 
 25  axlespacings      object 
dtypes: float64(7),

# Cheaking for null values

In [ ]:
taa.isnull().sum()

,0
cosit,0
year,0
month,0
day,0
hour,0
minute,0
second,0
millisecond,0
minuteofday,0
lane,0


# Handling Null Values

In [ ]:
taa = taa.drop(columns=['axleweights','straddlelanename','axlespacings'])

We observerd that these three columns had many missing values and dropping them would not affect our visualization as it not related to what we are trying to find.

# Cleaning the column names

In [ ]:
taa.columns = taa.columns.str.lower().str.strip()
taa.columns

Index(['cosit', 'year', 'month', 'day', 'hour', 'minute', 'second',
       'millisecond', 'minuteofday', 'lane', 'lanename', 'straddlelane',
       'class', 'classname', 'length', 'headway', 'gap', 'speed', 'weight',
       'temperature', 'duration', 'validitycode', 'numberofaxles'],
      dtype='object')

Here we have converted all column names into lowercase and removed any extra spaces for saferside and for easier access to columns later in the code

# Mapping Vehicle Class Codes

In [ ]:
class_map = {
    0: "Unclassified",
    1: "Motorcycle",
    2: "Car",
    3: "Van / LGV",
    4: "Bus / Minibus",
    5: "Rigid HGV",
    6: "Articulated HGV",
    7: "Multi-Axle HGV"
}

taa['classname'] = taa['class'].map(class_map)
taa['classname'] = taa['classname'].fillna("Unknown")


The dataset stores vehicle types as numbers i.e from 0 to 7.

Therefore we converted those numeric code into meaningful names like Car,Bus etc..

# Creating a Date-Time Column

In [ ]:
taa['datetime']=pd.to_datetime(taa[['year', 'month', 'day', 'hour', 'minute', 'second']],errors='coerce')
taa['hour'] = taa['datetime'].dt.hour
taa['hour'].value_counts().sort_index().head()

,count
hour,
0,60677
1,38685
2,27831
3,26486
4,33118


We combined the year,month,day,hour,minute,second columns into a single DateTime column.

This is because we need to analyze the traffic trends over different hours of the day.

# Handling Missing Values

In [ ]:
clean = taa.dropna(subset=['hour','classname']).copy()
clean.shape

(4769072, 24)

Rows with missing hour or vehicle class were removed in this step.

# Grouping Traffic Counts by Hour and Vehicle Type

In [ ]:
agg_hour = clean.groupby(['hour','classname']).size().reset_index(name='count')
agg_hour.head(10)

,hour,classname,count
0,0,Articulated HGV,3305
1,0,Bus / Minibus,437
2,0,Car,51572
3,0,Motorcycle,248
4,0,Multi-Axle HGV,259
5,0,Rigid HGV,851
6,0,Unclassified,52
7,0,Van / LGV,3953
8,1,Articulated HGV,3214
9,1,Bus / Minibus,289


We have grouped the cleaned data to count how many of each vehicle type appeared every hour which is needed for analyzing hourly traffic patterns.

# Pivot Table

In [ ]:
pivot = agg_hour.pivot(index='hour', columns='classname', values='count').fillna(0)
pivot = pivot.reindex(range(0,24), fill_value=0)
pivot.head()

classname,Articulated HGV,Bus / Minibus,Car,Motorcycle,Multi-Axle HGV,Rigid HGV,Unclassified,Van / LGV
hour,,,,,,,,
0,3305,437,51572,248,259,851,52,3953
1,3214,289,31354,117,157,711,22,2821
2,2659,233,21503,78,50,692,13,2603
3,2623,295,19722,95,80,830,17,2824
4,2900,335,24732,196,67,1029,26,3833


We have created a pivot table for visualizations where -
* Rows represent the hours from 0 to 23

* Columns represent the Vehicle Types

* Values represent the number of vehicles



In [ ]:
pivot['total'] = pivot.sum(axis=1)

class_order = pivot.drop(columns='total').sum().sort_values(ascending=False).index.tolist()
class_order

['Car',
 'Van / LGV',
 'Articulated HGV',
 'Rigid HGV',
 'Motorcycle',
 'Multi-Axle HGV',
 'Bus / Minibus',
 'Unclassified']

We have also added a total traffic column for each hour.

# Exploratory Graph 1 : Total Vehicles per Class

In [ ]:
class_counts = clean['classname'].value_counts().reset_index()
class_counts.columns = ['classname','count']

fig1 = px.bar(
    class_counts,
    x='classname',
    y='count',
    title='Exploratory Graph 1: Total Vehicles per Class',
    color='classname'
)

fig1.update_layout(xaxis_tickangle=45, width=900, height=500)
fig1.show()

This bar chart shows how many vehicles are there in each class. This helps us to see which category dominates the overall traffic.

Note : This is not the final graph

# Exploratory Graph 2 : Total Traffic by Hour

In [ ]:
hourly_total = clean.groupby('hour').size().reset_index(name='total')

fig2 = px.line(
    hourly_total,
    x='hour',
    y='total',
    markers=True,
    title='Exploratory Graph 2: Total Traffic Volume by Hour',
    labels={'hour': 'Hour of Day', 'total': 'Vehicle Count'}
)

fig2.update_layout(width=900, height=500, xaxis=dict(dtick=1))
fig2.show()

This line chart shows how traffic volume changes throughout the day. This helps to identify busy and quite hours on road.

Note : This is not the final graph

# Final Visualization : Traffic Trend by Hour and Vehicle Class

In [ ]:
fig = go.Figure()

for cls in class_order:
    fig.add_trace(go.Scatter(
        x=pivot.index,
        y=pivot[cls],
        mode='lines+markers',
        name=str(cls),
        opacity=0.7,
        line=dict(width=2)
    ))

fig.add_trace(go.Scatter(
    x=pivot.index,
    y=pivot['total'],
    mode='lines+markers',
    name='Total Traffic',
    line=dict(width=4, color='black'),
    marker=dict(size=6)
))

fig.update_layout(
    title="Final Explanatory Graph: Hourly Traffic by Vehicle Class with Total Traffic Line",
    xaxis=dict(title='Hour of Day', dtick=1),
    yaxis=dict(title='Vehicle Count'),
    width=1200,
    height=650
)

fig.show()


This graph shows hourly traffic for each vehicle type also a bold line representing total traffic which helps us to compare which vehicle class contributes most at differnt hours.

# Peak Traffic Analysis

In [ ]:
peak_hour = int(pivot['total'].idxmax())
peak_total = int(pivot['total'].max())
dominant_class = pivot.loc[peak_hour].drop('total').idxmax()

print(f"Peak Hour: {peak_hour}:00 with {peak_total} vehicles")
print(f"Dominant Class at Peak: {dominant_class}")


Peak Hour: 12:00 with 398415 vehicles
Dominant Class at Peak: Car


We have calculated the peak hour of the day, total vehicles at that hour and vehicle class with highest contibution manually to give us a clear conclusion of when and what vehicles dominate traffic and confirmation that our visualization is correct.